In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DB_PATH = Path("../data/wihh.db")
conn = sqlite3.connect(DB_PATH)

# Sanity check
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)["name"].tolist()
print("Tables:", tables)

In [ ]:
# Berapa hari per bulan? Adakah gap?
coverage = pd.read_sql("""
    SELECT
        substr(date_wib, 1, 7) AS month,
        COUNT(*) AS n_days,
        SUM(complete) AS n_complete,
        ROUND(AVG(n_obs), 1) AS avg_obs_per_day
    FROM daily_summary
    GROUP BY month
    ORDER BY month
""", conn)
print(coverage)

In [ ]:
df = pd.read_sql("""
    SELECT date_wib, tmax, tmin, hours_rain, hours_ts, first_rain_hour
    FROM daily_summary
    WHERE complete = 1
""", conn)
df["date_wib"] = pd.to_datetime(df["date_wib"])
df["month"] = df["date_wib"].dt.month

# Distribusi Tmax keseluruhan
print("Tmax distribution:")
print(df["tmax"].value_counts().sort_index())

# Boxplot Tmax per bulan
df.boxplot(column="tmax", by="month", figsize=(12, 5))
plt.title("Tmax Jakarta (WIHH) per bulan")
plt.suptitle("")
plt.show()

In [ ]:
may_df = df[df["month"] == 5]
print(f"Sample size May: {len(may_df)} hari")
print(f"\nTmax distribution May:")
print(may_df["tmax"].value_counts(normalize=True).sort_index().round(3))

In [ ]:
# Hipotesis: hari yang hujannya datang lebih awal → Tmax lebih rendah
plt.figure(figsize=(10, 5))
plt.scatter(may_df["first_rain_hour"], may_df["tmax"], alpha=0.5)
plt.xlabel("First rain hour (WIB)")
plt.ylabel("Tmax (°C)")
plt.title("Mei: Timing of first rain vs Tmax")
plt.grid(alpha=0.3)
plt.show()

# Korelasi
print("Korelasi:", may_df[["first_rain_hour", "tmax"]].corr().iloc[0, 1])

In [ ]:
target_date = "2026-05-04"
day = pd.read_sql("""
    SELECT time_wib, temp_c, dewpoint_c, wx_phenomena, has_cb
    FROM metar_observations
    WHERE date_wib = ?
    ORDER BY time_wib
""", conn, params=[target_date])
print(f"\n{target_date} kronologi:")
print(day)

summary = pd.read_sql(
    "SELECT * FROM daily_summary WHERE date_wib = ?",
    conn, params=[target_date]
)
print(f"\nSummary: Tmax={summary['tmax'].iloc[0]} at {summary['tmax_time_wib'].iloc[0]}")